In [1]:
#! /usr/bin/env python


#==================
# Script to download Sentinel data (1, 2 or 3) in the region of interest defined by buoys positions (using buoy_date_region module) and sea ice edge from MW products (using ice_open module)

# authors: Anastasia Tarasenko (& peps team for S1 downloading)
# creation: oct 2016
#==================

import os, sys
# sys.path.append('/home/cercache/users/atarasen/codes/python/collocation')
sys.path.append('/home/cercache/users/atarasen/codes/python/')
sys.path.append('/home3/homedir7/perso/atarasen/codes/python/collocation')
import numpy as np
import nice_functions as nf
import numpy.ma as ma
import pickle
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from mpl_toolkits.basemap import Basemap

import datetime

import buoy as buoy_module
reload(buoy_module)
import ice as ice_module
import ncdfView as nv


def date_list(t1, t2):
  numdays = (t2-t1).days
  dt_list = [t1 + datetime.timedelta(days=1) for x in range(0, numdays)]
  return dt_list	
  
def interpolate_ice(lon_ice, lat_ice, lat_x, lon_x, area_lon=0.5, area_lat=0.5):
#     ind_area = np.where(np.logical_and(np.abs(lon_ice-lon_x)<area_lon, np.abs(lat_ice-lat_x)<area_lat))

    ind_area0, ind_area1 = np.where(np.logical_and(np.abs(lon_ice-lon_x)<area_lon, np.abs(lat_ice-lat_x)<area_lat))
    
#     print ind_area0

    # Enlarge the area of search, if no data was found  
    area_n=area_lat*2
    while (not ind_area0.any()):
        ind_area0, ind_area1 = np.where(np.logical_and(np.abs(lon_ice-lon_x)<(area_n), np.abs(lat_ice-lat_x)<(area_lat*area_n)))
        area_n=area_n+area_lat
        
    
    l=[]
    for i_count, ind0 in enumerate(ind_area0):
        ind1 = ind_area1[i_count]
          
        x=lon_ice[ind0, ind1]-lon_x
        y=lat_ice[ind0, ind1]-lat_x
        l.append(np.sqrt(x**2 +y**2))

    ind_ind = l.index(np.min(l))  
    true_ind = [ind_area0[ind_ind], ind_area1[ind_ind]] 
    return true_ind    
    

No extra plotting objects loaded...


/usr/local/lib/python2.7/dist-packages/matplotlib/__init__.py:1357: UserWarning:  This call to matplotlib.use() has no effect
because the backend has already been chosen;
matplotlib.use() must be called *before* pylab, matplotlib.pyplot,
or matplotlib.backends is imported for the first time.

  warnings.warn(_use_error_msg)


In [2]:
# Choose which mask to apply ('OSISAF', which starts from Sep, 21, 2015  or recompute from 'SSMI')
case_ice='SSMI'

# Choose

# case_list=['EastSiberian_Chukchi']
# case_list=['Svalbard']
case_list=['Greenland'] 
# case_list=['Kara', 'Canadian', 'CentralArctic', 'EastSiberian_Chukchi',]
# case_list=['Kara','Greenland']

# time interval for finding the minmax coordinate around buoy, in days

for case in case_list:
    fpath='/home/cercache/users/atarasen/data/Buoys/Arctic-buoy-data/'+case+'/' 
    fdir=os.listdir(fpath)
    print fdir
  
    for fname in fdir:
        if fname[-4:]=='.dat':
#         if fname=='itp76rawlocs.dat':
            
            print fname
            
#       check if final directories exist
            fpathout_fig='/home3/homedir7/perso/atarasen/Figures'
            fpathout_base='/home/cercache/users/atarasen/data/Sentinel-1_analysis/S1/'+case+'/'
            if os.path.isdir(fpathout_base)==False:
                os.system('mkdir %s' %(fpathout_base))

            fpathout='/home/cercache/users/atarasen/data/Sentinel-1_analysis/S1/'+case+'/'+fname[:-4]
            if os.path.isdir(fpathout)==False:
                os.system('mkdir %s' %(fpathout))
                print fpathout, 'is created'
#                 nf.pause()


#       import initial time series of the min-max coordinates around buoy + t1 and t2 of these "time windows"
            nf.tic()
            year, day, lon, lat = buoy_module.importtext(fpath, fname)
            lats, lons, days = buoy_module.buoy_daily(year, day, lon, lat)
                    
            
            true_ind0=[]; true_ind1=[]
            lat_b=[]; lon_b=[]
            count_miz=0
            
            for n in range(0,len(days)):
#                 print fname
                print n, 'from', len(days)
                day_str=days[n].strftime('%Y-%m-%d')
                if case_ice=='SSMI':
#                     print 'case SSMI', day_str

                    try:
# open appropriate SSMI sea ice concentration mask (ice - concentration, with negative values over the land (-128), sea ice concentration is from 0 to 100, 
                        ice, lon_ice, lat_ice = ice_module.SSMI_ice_open(day_str)
                    except:
                        pass
    
#     if the file with ice information is NOT empty
                    if ice!=[]:
                        ice_data=np.squeeze(ice)	#make the masked variable readable 2d array with filled data
#                         print 'Ice data shape', ice_data.shape                       
#                         print 'lats[n], lons[n]', lats[n], lons[n]


# have to skip part of trajectory, cause there is no data in the 'pole hole: for SSMI instrument:
                        if lats[n] < 87.2:
                            true_ind_n = interpolate_ice(lon_ice, lat_ice, lats[n], lons[n], 0.1, 0.1)

                            true_ind0.append(true_ind_n[0])
                            true_ind1.append(true_ind_n[1])
                    
                            lat_b.append(lat_ice[true_ind_n[0]])    
                            lon_b.append(lon_ice[true_ind_n[1]])                     
                    
    
# compare with MW data pixel by pixel
                            percent=30
                            area=12      # enlarge area by 4 pixels = 50 km from each side (distance btwn neigbough 
                                        # pixels is 12.5 km)
                            open_water_area=False
                            ice_area=False
                        
                            buoy_area=np.zeros((area+1, area+1)) 
                            buoy_area_lon=np.zeros((area+1, area+1)); buoy_area_lat=np.zeros((area+1, area+1))
                
                            for i_count, ii in enumerate(np.arange(-1*area/2, area/2+1)):
                                for j_count, jj in enumerate(np.arange(-1*area/2, area/2+1)):
                                    buoy_area_lon[i_count, j_count]=lon_ice[true_ind_n[0]+ii, true_ind_n[1]+jj]
                                    buoy_area_lat[i_count, j_count]=lat_ice[true_ind_n[0]+ii, true_ind_n[1]+jj]  
                                    buoy_area[i_count, j_count]=ice_data[true_ind_n[0]+ii, true_ind_n[1]+jj]
                                    
                                    if (buoy_area[i_count, j_count]<=percent)==True:
#                                         print ('Open water (C<%s) is found around the bouy!'%(percent))
                                        open_water_area=True
                                    elif np.logical_and(buoy_area[i_count, j_count]>=percent, \
                                                        buoy_area[i_count, j_count]<=100)==True:
#                                         print ('sea ice is found around the bouy, yuppi!')   
                                        ice_area=True                                     
                                        
                            if open_water_area==True and ice_area==True:
                                print ('we are not far from MIZ!')
                                count_miz=count_miz+1
                                lonmin_s=str(int(np.min(buoy_area_lon))); 
                                lonmax_s=str(int(np.ceil(np.max(buoy_area_lon))))
                                
                                if lonmax_s=='360':
                                    lonmin_s='0'; lonmax_s='1'
                                    
                                latmin_s=str(int(np.min(buoy_area_lat))); 
                                latmax_s=str(int(np.ceil(np.max(buoy_area_lat))))
                                
                                print lonmin_s, lonmax_s, latmin_s, latmax_s
                                
                                
                                
                                print 'download S1 data with peps'
                                print 'fpathout',fpathout, 
                                print 'lonmin_s', lonmin_s,  'lonmax_s',lonmax_s
                                print 'latmin_s', latmin_s, 'latmax_s', latmax_s, 
                                print 'day_str', day_str, 'day_str', day_str
                                os.system('/home/cercache/users/atarasen/codes/python/peps/peps_download.py -w %s -c S1 --lonmin %s --lonmax %s --latmin %s --latmax %s -a /home/cercache/users/atarasen/codes/python/peps/peps.txt -d %s -f %s'%(fpathout, lonmin_s, lonmax_s, latmin_s, latmax_s, day_str, day_str))
                                
                                

['300234060179710.dat', '300234062888920.dat', '300234062888920.png', 'itp76_description.txt', 'itp76_description.txt~', 'Nouveau dossier', 'itp76grddata.tar.Z', 'itp76dat2.jpg', 'itp76dat1.jpg', 'itp76loc2.jpg', 'itp76rawlocs-lat-lon-days.pickle', 'itp83rawlocs-lat-lon-days.pickle', '300234060179710-lat-lon-days.pickle', '300234062888920-lat-lon-days.pickle', '300234060179710.png', 'itp83rawlocs-lat-lon-time.pickle', 'itp83dat3.jpg', 'itp83rawlocs.dat', 'itp83loc2.jpg', 'itp83loc1.jpg', 'itp83-lat-lon-time.pickle', 'itp83grddata.tar.Z', 'itp76rawlocs.dat', 'itp76loc1.jpg', 'itp76-lat-lon-time.pickle', 'itp76dat3.jpg']
300234060179710.dat
0 from 1132
/home/cercache/users/atarasen/data/SSMI/2014/20140110.nc
1 from 1132


/usr/local/lib/python2.7/dist-packages/numpy/ma/core.py:4185: UserWarning: Warning: converting a masked element to nan.
  warnings.warn("Warning: converting a masked element to nan.")


/home/cercache/users/atarasen/data/SSMI/2014/20140111.nc
2 from 1132
/home/cercache/users/atarasen/data/SSMI/2014/20140112.nc
3 from 1132
/home/cercache/users/atarasen/data/SSMI/2014/20140113.nc
we are not far from MIZ!
346 352 63 66
download S1 data with peps
fpathout /home/cercache/users/atarasen/data/Sentinel-1_analysis/S1/Greenland/300234060179710 lonmin_s 346 lonmax_s 352
latmin_s 63 latmax_s 66 day_str 2014-01-13 day_str 2014-01-13
4 from 1132
/home/cercache/users/atarasen/data/SSMI/2014/20140114.nc
5 from 1132
/home/cercache/users/atarasen/data/SSMI/2014/20140115.nc
6 from 1132
/home/cercache/users/atarasen/data/SSMI/2014/20140116.nc
7 from 1132
/home/cercache/users/atarasen/data/SSMI/2014/20140117.nc
8 from 1132
/home/cercache/users/atarasen/data/SSMI/2014/20140118.nc
9 from 1132
/home/cercache/users/atarasen/data/SSMI/2014/20140119.nc
10 from 1132
/home/cercache/users/atarasen/data/SSMI/2014/20140120.nc
11 from 1132
/home/cercache/users/atarasen/data/SSMI/2014/20140121.nc
12 f

In [ ]:
#                                             os.system('/home/cercache/users/atarasen/codes/python/Sentinel_download/Sentinel_download.py --latmin %s --latmax %s --lonmin %s --lonmax %s -d %s -f %s -a /home/cercache/users/atarasen/codes/python/Sentinel_download/apihub.txt -w %s -s S1A*GRDM'\
		       #%(latmin_s, latmax_s, lonmin_s, lonmax_s, start_d, end_d, fpathout))
		     #os.system('/home/cercache/users/atarasen/codes/python/Sentinel_download/Sentinel_download.py --latmin %s --latmax %s --lonmin %s --lonmax %s -d %s -f %s -a /home/cercache/users/atarasen/codes/python/Sentinel_download/apihub.txt -n -s S1A*GRDM'\
		       #%(latmin_s, latmax_s, lonmin_s, lonmax_s, start_d, end_d))                    
                    

                    
                            if n==len(days)-1:
                            
#                                 m = Basemap(projection='npstere',boundinglat=60,lon_0=0,resolution='l')

#                                 x, y = m(lon_ice, lat_ice)
#                                 x_b=[]; y_b=[]
#                                 for i, i_lat in enumerate(lat_b):
#                                     x_b_i, y_b_i = m(lat_b[i],lon_b[i])
#                                     x_b.append(x_b_i)
#                                     y_b.append(y_b_i)
#                                 x_b, y_b = m(lon_b, lat_b)

#                                 m.drawcoastlines()
#                                 m.fillcontinents(color='white',lake_color='white')
#                                 m.drawparallels(np.arange(-80.,81.,20.))
#                                 m.drawmeridians(np.arange(-180.,181.,20.))
#                                 m.drawmapboundary(fill_color='white')

#                                 m.pcolor(x, y, ice_data, shading='flat')
#                                 m.plot(x_b, y_b,  'k*')
#                                 plt.title(day_str)

#                                 ax = plt.gca()
#                                 divider = make_axes_locatable(ax)
#                                 cax = divider.append_axes("right", size="5%", pad=0.05)
#                                 plt.colorbar(ice_fig, cax=cax)
#                                 plt.clim(0,100)
                            
#                                 plt.savefig(fpathout_fig+'/'+fname+'.png', dpi=300)
#                                 plt.show()
                            
                            
                            
                            
                                fig=plt.figure(1, figsize=(9,24), dpi=300)
                                ice_fig=plt.imshow(ice_data)
                                plt.plot(true_ind1,true_ind0, 'k*')
#                                 plt.colorbar(ice_fig, orientation='vertical')
                                
                                ax = plt.gca()
                                divider = make_axes_locatable(ax)
                                cax = divider.append_axes("right", size="5%", pad=0.05)
                                plt.colorbar(ice_fig, cax=cax)
                                plt.clim(0,100)
                                
                                fig.tight_layout()
                                plt.savefig(fpathout_fig+'/'+day_str+'.png', dpi=300)
                                plt.show()
                                
                                nf.pause()
                                plt.close()

            nf.toc()
#             nf.pause()
            

In [ ]:
area_n=[]
n=0
while (bool(area_n)==False):
# while (area_n==[]):    
    print 'bebebe'
    n=n+1
    if n==3:
        break
#     area_n=[10]

In [ ]:
fig=plt.figure(1,figsize=(9,12), dpi=300)

# setup north polar stereographic basemap.
# The longitude lon_0 is at 6-o'clock, and the
# latitude circle boundinglat is tangent to the edge
# of the map at lon_0. Default value of lat_ts
# (latitude of true scale) is pole.
m = Basemap(projection='npstere',boundinglat=60,lon_0=0,resolution='l')

x, y = m(lon_ice, lat_ice)

# lat_buoy, lon_buoy=np.meshgrid(lat_b,lon_b)

x_b=[]; y_b=[]
for i, i_lat in enumerate(lat_b):
    x_b_i, y_b_i = m(lat_b[i],lon_b[i])
    x_b.append(x_b_i)
    y_b.append(y_b_i)

m.drawcoastlines()
m.fillcontinents(color='white',lake_color='white')
# draw parallels and meridians.
m.drawparallels(np.arange(-80.,81.,20.))
m.drawmeridians(np.arange(-180.,181.,20.))
m.drawmapboundary(fill_color='white')

m.pcolor(x, y, ice_data, shading='flat')
m.plot(x_b, y_b,  'k*')
plt.title(day_str)

ax = plt.gca()
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.3)
plt.colorbar(ice_fig, cax=cax)
plt.clim(0,100)



# plt.title("North Polar Stereographic Projection")
plt.show()



In [ ]:
                    area=4      #enlarge area by 4 pixels = 50 km from each side (distance btwn neibough pixels is 12.5 km)
                    print true_ind_n[0]
                    buoy_area=np.zeros((area+1, area+1))
            
                    for i_count, ii in enumerate(np.arange(-1*area/2, area/2+1)):
                        for j_count, jj in enumerate(np.arange(-1*area/2, area/2+1)):
                            buoy_area[i_count, j_count]=lat_ice[true_ind_n[0]+ii, true_ind_n[1]+jj]

In [ ]:
    
# compare with MW data pixel by pixel
                    percent=15
                    area=4      #enlarge area by 4 pixels = 50 km from each side (distance btwn neibough pixels is 12.5 km)
                    
                    buoy_area=np.zeros((area+1, area+1))
                    for i_count, ii in enumerate(np.arange(-1*area/2, area/2+1)):
                        for j_count, jj in in enumerate(np.arange(-1*area/2, area/2+1)):
                            buoy_area[i_count, j_count]=ice_data[true_ind_n[0]+ii, true_ind_n[1]+jj]
                            
            
                    for i_pixel in range(0, len(buoy_area):
                        for i_pixel in range(0, len(buoy_area):
                            ice_pixel=ice_data[pixel[0][indb_i],ind_buoy_area[1][indb_i]]
                        
                        if np.logical_and(ice_pixel>=percent, ice_pixel<=100)==True:
                      print ('sea ice (C>%s) is found around the bouy, yuppi!'%(percent))
	          
                      for indb_w in range(0, len(ind_buoy_area[0])):
                        if np.logical_and(ice_pixel>=0, ice_pixel<=percent)==True:
		     print 'oh! and open water as well is found around the bouy! We are in MIZ!' 
	             start_d=t1_list[n].strftime('%Y-%m-%d')	#start_d should be in format '2015-11-01'
	             end_d=t2_list[n].strftime('%Y-%m-%d')
	             latmin_s=str(int(latmin[n]))
	             latmax_s=str(int(latmax[n]))
	             lonmin_s=str(int(lonmin[n]))
	             lonmax_s=str(int(lonmax[n]))
	             print latmin[n], latmin_s, latmax_s, lonmin[n], lonmin_s, lonmax_s
	             
	            #Visualize
	            
	             fig=plt.figure(1, figsize=(9,24), dpi=300)
	             ice_fig=plt.imshow(np.transpose(ice_data))
	             plt.axis([0,900, 0, 600])
	             plt.plot(ind_buoy_area[0],ind_buoy_area[1], 'k*')
	             plt.title('Sea ice concentration\n and buoy area", %s-%s'%(start_d, end_d))
	             plt.colorbar(ice_fig, orientation='vertical')
	             fig.tight_layout()
                     plt.show()
                     plt.savefig(fpathout+'/'+day_str+'.png', dpi=300)
                     
	             #print 'download S1 data with peps'
	             #os.system('/home/cercache/users/atarasen/codes/python/peps/peps_download.py -w %s -c S1 --lonmin %s --lonmax %s --latmin %s --latmax %s -a /home/cercache/users/atarasen/codes/python/peps/peps.txt -d %s -f %s'%(fpathout, lonmin_s, lonmax_s, latmin_s, latmax_s, start_d, end_d))
		     #os.system('/home/cercache/users/atarasen/codes/python/Sentinel_download/Sentinel_download.py --latmin %s --latmax %s --lonmin %s --lonmax %s -d %s -f %s -a /home/cercache/users/atarasen/codes/python/Sentinel_download/apihub.txt -w %s -s S1A*GRDM'\
		       #%(latmin_s, latmax_s, lonmin_s, lonmax_s, start_d, end_d, fpathout))
		     #os.system('/home/cercache/users/atarasen/codes/python/Sentinel_download/Sentinel_download.py --latmin %s --latmax %s --lonmin %s --lonmax %s -d %s -f %s -a /home/cercache/users/atarasen/codes/python/Sentinel_download/apihub.txt -n -s S1A*GRDM'\
		       #%(latmin_s, latmax_s, lonmin_s, lonmax_s, start_d, end_d))
               
                     break		
	
		
	  # OSISAF mask has 0.1 degree resolution and mask(time, lat, lon) variable should be equal to 3 (or to 9???...) to represent the ice presence	    
	  elif case_ice=='OSISAF':    
	    ice_mask, lon_ice, lat_ice=ice_open.osisaf_icemask_open(start_d)
	    
	    latmin_01deg=round(latmin[n],1)
	    latmax_01deg=round(latmax[n],1)
	    lats_0=np.arange(latmin_01deg, latmax_01deg, 0.1)
	    
	    lonmin_01deg=round(lonmin[n],1)
	    lonmax_01deg=round(lonmax[n],1)
	    lons_0=np.arange(lonmin_01deg, lonmax_01deg, 0.1)
	    
	    # Create a small 2D square area to look at the ice mask (where the the exact buoy position is in the center)
	    lons_2D, lats_2D=np.meshgrid(lons_0,lats_0)
	    # Reshape it to 1D for the sake of calculation rapidity
	    lons_1D=np.reshape(lons_2D, np.shape(lons_2D)[0]*np.shape(lons_2D)[1])
	    lats_1D=np.reshape(lats_2D, np.shape(lats_2D)[0]*np.shape(lats_2D)[1])
	    
	    for i_coord in range(0,len(lons_1D)):
	      ice_ind=ice_mask[lat_ice==lats_1D[i_coord], lon_ice==lons_1D[i_coord]]
	      if  ice_ind==9:
	        latmin_s=str(int(latmin_01deg))
	        latmax_s=str(int(latmax_01deg))
	        lonmin_s=str(int(lonmin_01deg))
	        lonmax_s=str(int(lonmax_01deg))
	        
	        print 'download S1 data with peps'
	        os.system('/home/cercache/users/atarasen/codes/python/peps/peps_download.py -w %s -c S1 --lonmin %s --lonmax %s --latmin %s --latmax %s -a /home/cercache/users/atarasen/codes/python/peps/peps.txt -d %s -f %s'%(fpathout, lonmin_s, lonmax_s, latmin_s, latmax_s, start_d, end_d))
                break
      
    
  #fname='135384-lat-lon-time.pickle'
  #with open(fpath+fname) as f:
    #latmin, latmax, lonmin, lonmax, start_date, end_date = pickle.load(f)